In [1]:
import torch
import time
import psutil
import plotly.graph_objects as go
from scipy.stats import wasserstein_distance

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define a 1D Gaussian Mixture Model (GMM)
def log_prob(x):
    """Compute log probability of a Gaussian mixture model."""
    x = x.to(device).requires_grad_(True)
    p1 = torch.exp(-0.5 * ((x - 2) / 0.8) ** 2) / (0.8 * (2 * torch.pi) ** 0.5)
    p2 = torch.exp(-0.5 * ((x + 2) / 0.8) ** 2) / (0.8 * (2 * torch.pi) ** 0.5)
    return torch.log(0.5 * p1 + 0.5 * p2 + 1e-9)  # Small constant for numerical stability

# Compute the true score function using autograd
def true_score(x, step, step_size):
    x = x.to(device)
    log_p = log_prob(x)
    grad = step_size * torch.autograd.grad(log_p.sum(), x, create_graph=True)[0]
    return grad.detach(), step_size

# Estimate the score function using finite differences
def estimate_score(x, step, step_size, delta=5e-2):
    x = x.to(device)
    step_size = (1 + step_size) ** 0.6
    delta = step_size * torch.tensor(delta / (step + 1) ** 0.5, device=device)  # Adaptive step size
    return (log_prob(x + delta) - log_prob(x - delta)) / (2 * delta), step_size

# Measure memory usage on CUDA
def get_memory_usage():
    if torch.cuda.is_available():
        torch.cuda.synchronize()  # Ensure all GPU operations are completed
        return torch.cuda.max_memory_allocated() / (1024**2)  # Peak memory in MB
    else:
        return psutil.Process().memory_info().rss / (1024**2)  # CPU memory usage

# Langevin Dynamics sampler with better exploration
def langevin_dynamics(x0_list, score_function, steps=100, eta=0.5, noise_scale=0.7):
    """Perform Langevin Dynamics using a specified score function, with multiple initial points."""
    x = x0_list.to(device)
    samples = []

    for i in range(steps):
        score, step_size = score_function(x, i, eta)  # Compute score function
        noise = torch.randn_like(x, device=device) * noise_scale  # Reduce noise over time
        x = x + score + torch.sqrt(torch.tensor(2 * step_size, device=device)) * noise
        samples.append(x.clone().detach())

    return torch.cat(samples)

# Experiment configurations
num_samples_list = [1, 10, 100, 1000, 10000]
steps_list = [1, 10, 100, 1000, 10000]
results = []

# Run experiments
for num_samples in num_samples_list:
    for steps in steps_list:
        x0_list = torch.randn(num_samples, device=device)
        
        # Benchmarking True Gradients
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()  # Reset memory tracking
        start_time = time.perf_counter()
        mem_before = get_memory_usage()
        samples_true = langevin_dynamics(x0_list, true_score, steps)
        torch.cuda.synchronize()
        mem_after = get_memory_usage()
        time_true = time.perf_counter() - start_time
        mem_true = mem_after - mem_before

        # Benchmarking Finite Differences
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()  # Reset memory tracking
        start_time = time.perf_counter()
        mem_before = get_memory_usage()
        samples_estimated = langevin_dynamics(x0_list, estimate_score, steps)
        torch.cuda.synchronize()
        mem_after = get_memory_usage()
        time_estimated = time.perf_counter() - start_time
        mem_estimated = mem_after - mem_before

        # Compute Wasserstein distance
        samples_true_np = samples_true.detach().cpu().numpy()
        samples_estimated_np = samples_estimated.detach().cpu().numpy()
        x_vals = torch.linspace(-5, 5, 1000).to(device)
        true_density = torch.exp(log_prob(x_vals)).detach().cpu().numpy()

        wasserstein_true = wasserstein_distance(samples_true_np.flatten(), true_density)
        wasserstein_estimated = wasserstein_distance(samples_estimated_np.flatten(), true_density)

        # Collect results
        results.append({
            "num_samples": num_samples,
            "steps": steps,
            "time_true": time_true,
            "mem_true": mem_true,
            "time_estimated": time_estimated,
            "mem_estimated": mem_estimated,
            "wasserstein_true": wasserstein_true,
            "wasserstein_estimated": wasserstein_estimated
        })
        print(f"Completed: num_samples = {num_samples}, steps = {steps}")

# Prepare data for plotting
def prepare_plot_data(results, variable):
    num_samples = sorted(list(set(r["num_samples"] for r in results)))
    steps = sorted(list(set(r["steps"] for r in results)))
    data_true = [[r[f"{variable}_true"] for r in results if r["steps"] == s and r["num_samples"] == num_samples] for s in steps]
    data_estimated = [[r[f"{variable}_estimated"] for r in results if r["steps"] == s and r["num_samples"] == num_samples] for s in steps]
    return num_samples, steps, data_true, data_estimated

def plot_3d(num_samples, steps, data_true, data_estimated, ylabel, title, filename):
    fig = go.Figure()
    fig.add_trace(go.Surface(z=data_true, x=num_samples, y=steps, colorscale='Viridis', name='True Gradients'))
    fig.add_trace(go.Surface(z=data_estimated, x=num_samples, y=steps, colorscale='Cividis', name='Finite Differences', showscale=False))
    fig.update_layout(
        scene=dict(
            xaxis_title='Number of Samples',
            yaxis_title='Number of Steps',
            zaxis_title=ylabel,
            xaxis_type='log',
            yaxis_type='log'
        ),
        title=title
    )
    fig.write_html(filename)
    print(f"Plot saved to {filename}")

# Plot results
num_samples, steps, data_true, data_estimated = prepare_plot_data(results, "time")
plot_3d(num_samples, steps, data_true, data_estimated, "Runtime (seconds)", "Runtime Comparison: True Gradients vs Finite Differences", "runtime_comparison.html")

num_samples, steps, data_true, data_estimated = prepare_plot_data(results, "mem")
plot_3d(num_samples, steps, data_true, data_estimated, "Memory Usage (MB)", "Memory Usage Comparison: True Gradients vs Finite Differences", "memory_comparison.html")

num_samples, steps, data_true, data_estimated = prepare_plot_data(results, "wasserstein")
plot_3d(num_samples, steps, data_true, data_estimated, "Wasserstein Distance", "Wasserstein Distance Comparison: True Gradients vs Finite Differences", "wasserstein_comparison.html")

Completed: num_samples = 1, steps = 1
Completed: num_samples = 1, steps = 10
Completed: num_samples = 1, steps = 100
Completed: num_samples = 1, steps = 1000
Completed: num_samples = 1, steps = 10000
Completed: num_samples = 10, steps = 1
Completed: num_samples = 10, steps = 10
Completed: num_samples = 10, steps = 100
Completed: num_samples = 10, steps = 1000


KeyboardInterrupt: 

In [2]:
# # Benchmarking
# num_samples = 100000
# x0_list = torch.randn(num_samples, device=device)
# steps = 100  # Increase the number of steps

# # Benchmarking True Gradients
# torch.cuda.empty_cache()
# torch.cuda.reset_peak_memory_stats()  # Reset memory tracking
# start_time = time.perf_counter()
# mem_before = get_memory_usage()
# samples_true = langevin_dynamics(x0_list, true_score, steps)
# torch.cuda.synchronize()
# mem_after = get_memory_usage()
# time_true = time.perf_counter() - start_time
# mem_true = mem_after - mem_before

# # Benchmarking Finite Differences
# torch.cuda.empty_cache()
# torch.cuda.reset_peak_memory_stats()  # Reset memory tracking
# start_time = time.perf_counter()
# mem_before = get_memory_usage()
# samples_estimated = langevin_dynamics(x0_list, estimate_score, steps)
# torch.cuda.synchronize()
# mem_after = get_memory_usage()
# time_estimated = time.perf_counter() - start_time
# mem_estimated = mem_after - mem_before

# # Compute Wasserstein distance
# samples_true_np = samples_true.detach().cpu().numpy()
# samples_estimated_np = samples_estimated.detach().cpu().numpy()
# x_vals = torch.linspace(-5, 5, 1000).to(device)
# true_density = torch.exp(log_prob(x_vals)).detach().cpu().numpy()

# wasserstein_true = wasserstein_distance(samples_true_np.flatten(), true_density)
# wasserstein_estimated = wasserstein_distance(samples_estimated_np.flatten(), true_density)

# # Print corrected memory usage and Wasserstein distances
# print(f"Runtime (True Gradients): {time_true:.4f} sec")
# print(f"Peak Memory Usage (True Gradients): {mem_true:.2f} MB")
# print(f"Runtime (Finite Differences): {time_estimated:.4f} sec")
# print(f"Peak Memory Usage (Finite Differences): {mem_estimated:.2f} MB")
# print(f"Wasserstein Distance (True Gradients): {wasserstein_true:.4f}")
# print(f"Wasserstein Distance (Finite Differences): {wasserstein_estimated:.4f}")

# # Plot runtime comparison
# plt.figure(figsize=(8, 5))
# plt.bar(["True Gradients", "Finite Differences"], [time_true, time_estimated], color=['blue', 'green'])
# plt.ylabel("Runtime (seconds)")
# plt.title("Runtime Comparison: True Gradients vs Finite Differences")
# plt.show()

# # Plot memory comparison
# plt.figure(figsize=(8, 5))
# plt.bar(["True Gradients", "Finite Differences"], [mem_true, mem_estimated], color=['blue', 'green'])
# plt.ylabel("Memory Usage (MB)")
# plt.title("Memory Usage Comparison: True Gradients vs Finite Differences")
# plt.show()

# # Plot histogram of samples
# plt.figure(figsize=(8, 5))
# plt.hist(samples_true_np, bins=100, density=True, alpha=0.5, label="Langevin (True Gradient)", color="blue")
# plt.hist(samples_estimated_np, bins=100, density=True, alpha=0.5, label="Langevin (Finite Difference)", color="green")
# plt.plot(x_vals.detach().cpu().numpy(), true_density, label="True Distribution", linewidth=2, color="red")
# plt.xlabel("x")
# plt.ylabel("Density")
# plt.legend()
# plt.title("Comparison: Langevin Sampling (True Gradient vs Finite Difference)")
# plt.show()

In [3]:
# Experiment configurations
num_samples_list = [1, 10, 100, 1000, 10000]
steps_list = [1, 10, 100, 1000, 10000]
results = []

# Run experiments
for num_samples in num_samples_list:
    for steps in steps_list:
        x0_list = torch.randn(num_samples, device=device)
        
        # Benchmarking True Gradients
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()  # Reset memory tracking
        start_time = time.perf_counter()
        mem_before = get_memory_usage()
        samples_true = langevin_dynamics(x0_list, true_score, steps)
        torch.cuda.synchronize()
        mem_after = get_memory_usage()
        time_true = time.perf_counter() - start_time
        mem_true = mem_after - mem_before

        # Benchmarking Finite Differences
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()  # Reset memory tracking
        start_time = time.perf_counter()
        mem_before = get_memory_usage()
        samples_estimated = langevin_dynamics(x0_list, estimate_score, steps)
        torch.cuda.synchronize()
        mem_after = get_memory_usage()
        time_estimated = time.perf_counter() - start_time
        mem_estimated = mem_after - mem_before

        # Compute Wasserstein distance
        samples_true_np = samples_true.detach().cpu().numpy()
        samples_estimated_np = samples_estimated.detach().cpu().numpy()
        x_vals = torch.linspace(-5, 5, 1000).to(device)
        true_density = torch.exp(log_prob(x_vals)).detach().cpu().numpy()

        wasserstein_true = wasserstein_distance(samples_true_np.flatten(), true_density)
        wasserstein_estimated = wasserstein_distance(samples_estimated_np.flatten(), true_density)

        # Collect results
        results.append({
            "num_samples": num_samples,
            "steps": steps,
            "time_true": time_true,
            "mem_true": mem_true,
            "time_estimated": time_estimated,
            "mem_estimated": mem_estimated,
            "wasserstein_true": wasserstein_true,
            "wasserstein_estimated": wasserstein_estimated
        })
        print(f"Completed: num_samples = {num_samples}, steps = {steps}")

Completed: num_samples = 1, steps = 1
Completed: num_samples = 1, steps = 10
Completed: num_samples = 1, steps = 100
Completed: num_samples = 1, steps = 1000


Completed: num_samples = 1, steps = 10000
Completed: num_samples = 10, steps = 1
Completed: num_samples = 10, steps = 10
Completed: num_samples = 10, steps = 100
Completed: num_samples = 10, steps = 1000
Completed: num_samples = 10, steps = 10000
Completed: num_samples = 100, steps = 1
Completed: num_samples = 100, steps = 10
Completed: num_samples = 100, steps = 100
Completed: num_samples = 100, steps = 1000
Completed: num_samples = 100, steps = 10000
Completed: num_samples = 1000, steps = 1
Completed: num_samples = 1000, steps = 10
Completed: num_samples = 1000, steps = 100
Completed: num_samples = 1000, steps = 1000
Completed: num_samples = 1000, steps = 10000
Completed: num_samples = 10000, steps = 1
Completed: num_samples = 10000, steps = 10
Completed: num_samples = 10000, steps = 100
Completed: num_samples = 10000, steps = 1000
Completed: num_samples = 10000, steps = 10000
Completed: num_samples = 10000, steps = 1
Completed: num_samples = 10000, steps = 10
Completed: num_samples 

In [4]:
# # Plot results
# def plot_results(results, variable, ylabel, title):
#     plt.figure(figsize=(10, 6))
#     for steps in steps_list:
#         data = [r for r in results if r["steps"] == steps]
#         plt.plot([r["num_samples"] for r in data], [r[variable] for r in data], label=f"Steps = {steps}")
#     plt.xscale("log")
#     plt.yscale("log")
#     plt.xlabel("Number of Samples")
#     plt.ylabel(ylabel)
#     plt.title(title)
#     plt.legend()
#     plt.grid(True)
#     plt.show()

# def plot_comparison(results, variable, ylabel, title):
#     plt.figure(figsize=(10, 6))
#     for steps in steps_list:
#         data = [r for r in results if r["steps"] == steps]
#         plt.plot([r["num_samples"] for r in data], [r[f"{variable}_true"] for r in data], label=f"True Gradients (Steps = {steps})", linestyle='--')
#         plt.plot([r["num_samples"] for r in data], [r[f"{variable}_estimated"] for r in data], label=f"Finite Differences (Steps = {steps})", linestyle='-')
#     plt.xscale("log")
#     plt.yscale("log")
#     plt.xlabel("Number of Samples")
#     plt.ylabel(ylabel)
#     plt.title(title)
#     plt.legend()
#     plt.grid(True)
#     plt.show()

# plot_comparison(results, "time", "Runtime (seconds)", "Runtime Comparison: True Gradients vs Finite Differences")
# plot_comparison(results, "mem", "Memory Usage (MB)", "Memory Usage Comparison: True Gradients vs Finite Differences")
# plot_comparison(results, "wasserstein", "Wasserstein Distance", "Wasserstein Distance Comparison: True Gradients vs Finite Differences")

In [6]:
# Prepare data for plotting
def prepare_plot_data(results, variable):
    num_samples = sorted(list(set(r["num_samples"] for r in results)))
    steps = sorted(list(set(r["steps"] for r in results)))
    data_true = [[r[f"{variable}_true"] for r in results if r["steps"] == s and r["num_samples"] == num_samples] for s in steps]
    data_estimated = [[r[f"{variable}_estimated"] for r in results if r["steps"] == s and r["num_samples"] == num_samples] for s in steps]
    return num_samples, steps, data_true, data_estimated

def plot_3d(num_samples, steps, data_true, data_estimated, ylabel, title, filename):
    fig = go.Figure()
    fig.add_trace(go.Surface(z=data_true, x=num_samples, y=steps, colorscale='Viridis', name='True Gradients'))
    fig.add_trace(go.Surface(z=data_estimated, x=num_samples, y=steps, colorscale='Cividis', name='Finite Differences', showscale=False))
    fig.update_layout(
        scene=dict(
            xaxis_title='Number of Samples',
            yaxis_title='Number of Steps',
            zaxis_title=ylabel,
            xaxis_type='log',
            yaxis_type='log'
        ),
        title=title
    )
    fig.write_html(filename)

# Plot results
num_samples, steps, data_true, data_estimated = prepare_plot_data(results, "time")
plot_3d(num_samples, steps, data_true, data_estimated, "Runtime (seconds)", "Runtime Comparison: True Gradients vs Finite Differences", "runtime_comparison.html")

num_samples, steps, data_true, data_estimated = prepare_plot_data(results, "mem")
plot_3d(num_samples, steps, data_true, data_estimated, "Memory Usage (MB)", "Memory Usage Comparison: True Gradients vs Finite Differences", "memory_comparison.html")

num_samples, steps, data_true, data_estimated = prepare_plot_data(results, "wasserstein")
plot_3d(num_samples, steps, data_true, data_estimated, "Wasserstein Distance", "Wasserstein Distance Comparison: True Gradients vs Finite Differences", "wasserstein_comparison.html")

In [9]:
import json
import plotly.express as px
import plotly.graph_objects as go

# Load results from file
with open('results.json', 'r') as f:
    results = json.load(f)
print("Results loaded from results.json")

# Prepare data for plotting
def prepare_plot_data(results, variable):
    num_samples = sorted(list(set(r["num_samples"] for r in results)))
    steps = sorted(list(set(r["steps"] for r in results)))
    data_true = {sample: [r[f"{variable}_true"] for r in results if r["num_samples"] == sample] for sample in num_samples}
    data_estimated = {sample: [r[f"{variable}_estimated"] for r in results if r["num_samples"] == sample] for sample in num_samples}
    data_spsa = {sample: [r[f"{variable}_spsa"] for r in results if r["num_samples"] == sample] for sample in num_samples}
    data_hmc_true = {sample: [r[f"{variable}_hmc_true"] for r in results if r["num_samples"] == sample] for sample in num_samples}
    data_hmc_estimated = {sample: [r[f"{variable}_hmc_estimated"] for r in results if r["num_samples"] == sample] for sample in num_samples}
    data_hmc_spsa = {sample: [r[f"{variable}_hmc_spsa"] for r in results if r["num_samples"] == sample] for sample in num_samples}
    return num_samples, steps, data_true, data_estimated, data_spsa, data_hmc_true, data_hmc_estimated, data_hmc_spsa

def plot_line_graph(data, steps, xlabel, ylabel, title, filename):
    fig = go.Figure()
    for num_samples, values in data.items():
        fig.add_trace(go.Scatter(x=steps, y=values, mode='lines', name=f'Num Samples: {num_samples}'))
    fig.update_layout(title=title, xaxis_title=xlabel, yaxis_title=ylabel)
    fig.write_html(filename)
    print(f"Plot saved to {filename}")

# Plot results
num_samples, steps, data_true, data_estimated, data_spsa, data_hmc_true, data_hmc_estimated, data_hmc_spsa = prepare_plot_data(results, "time")
plot_line_graph(data_true, steps, "Number of Steps", "Runtime (seconds)", "Runtime (seconds): True Gradients", "runtime_true.html")
plot_line_graph(data_estimated, steps, "Number of Steps", "Runtime (seconds)", "Runtime (seconds): Finite Differences", "runtime_estimated.html")
plot_line_graph(data_spsa, steps, "Number of Steps", "Runtime (seconds)", "Runtime (seconds): SPSA", "runtime_spsa.html")
plot_line_graph(data_hmc_true, steps, "Number of Steps", "Runtime (seconds)", "Runtime (seconds): HMC True", "runtime_hmc_true.html")
plot_line_graph(data_hmc_estimated, steps, "Number of Steps", "Runtime (seconds)", "Runtime (seconds): HMC Estimated", "runtime_hmc_estimated.html")
plot_line_graph(data_hmc_spsa, steps, "Number of Steps", "Runtime (seconds)", "Runtime (seconds): HMC SPSA", "runtime_hmc_spsa.html")

num_samples, steps, data_true, data_estimated, data_spsa, data_hmc_true, data_hmc_estimated, data_hmc_spsa = prepare_plot_data(results, "mem")
plot_line_graph(data_true, steps, "Number of Steps", "Memory Usage (MB)", "Memory Usage (MB): True Gradients", "memory_true.html")
plot_line_graph(data_estimated, steps, "Number of Steps", "Memory Usage (MB)", "Memory Usage (MB): Finite Differences", "memory_estimated.html")
plot_line_graph(data_spsa, steps, "Number of Steps", "Memory Usage (MB)", "Memory Usage (MB): SPSA", "memory_spsa.html")
plot_line_graph(data_hmc_true, steps, "Number of Steps", "Memory Usage (MB)", "Memory Usage (MB): HMC True", "memory_hmc_true.html")
plot_line_graph(data_hmc_estimated, steps, "Number of Steps", "Memory Usage (MB)", "Memory Usage (MB): HMC Estimated", "memory_hmc_estimated.html")
plot_line_graph(data_hmc_spsa, steps, "Number of Steps", "Memory Usage (MB)", "Memory Usage (MB): HMC SPSA", "memory_hmc_spsa.html")

num_samples, steps, data_true, data_estimated, data_spsa, data_hmc_true, data_hmc_estimated, data_hmc_spsa = prepare_plot_data(results, "wasserstein")
plot_line_graph(data_true, steps, "Number of Steps", "Wasserstein Distance", "Wasserstein Distance: True Gradients", "wasserstein_true.html")
plot_line_graph(data_estimated, steps, "Number of Steps", "Wasserstein Distance", "Wasserstein Distance: Finite Differences", "wasserstein_estimated.html")
plot_line_graph(data_spsa, steps, "Number of Steps", "Wasserstein Distance", "Wasserstein Distance: SPSA", "wasserstein_spsa.html")
plot_line_graph(data_hmc_true, steps, "Number of Steps", "Wasserstein Distance", "Wasserstein Distance: HMC True", "wasserstein_hmc_true.html")
plot_line_graph(data_hmc_estimated, steps, "Number of Steps", "Wasserstein Distance", "Wasserstein Distance: HMC Estimated", "wasserstein_hmc_estimated.html")
plot_line_graph(data_hmc_spsa, steps, "Number of Steps", "Wasserstein Distance", "Wasserstein Distance: HMC SPSA", "wasserstein_hmc_spsa.html")

Results loaded from results.json
Plot saved to runtime_true.html
Plot saved to runtime_estimated.html
Plot saved to runtime_spsa.html
Plot saved to runtime_hmc_true.html
Plot saved to runtime_hmc_estimated.html
Plot saved to runtime_hmc_spsa.html
Plot saved to memory_true.html
Plot saved to memory_estimated.html
Plot saved to memory_spsa.html
Plot saved to memory_hmc_true.html
Plot saved to memory_hmc_estimated.html
Plot saved to memory_hmc_spsa.html
Plot saved to wasserstein_true.html
Plot saved to wasserstein_estimated.html
Plot saved to wasserstein_spsa.html
Plot saved to wasserstein_hmc_true.html
Plot saved to wasserstein_hmc_estimated.html
Plot saved to wasserstein_hmc_spsa.html
